# 🚀 Phase 8: Export Module & Adaptive RLHF Systems

This notebook exports pages to PDF/CBZ/HTML and demonstrates the Human Alignment Telemetry Loop with parameter backpropagation optimization.

## 🔧 0. Universal Environment Setup
Run this cell first to configure Colab or local Jupyter environments.

In [ ]:
# ============================================================
# Universal Colab/Local Setup — run this first in every notebook
# ============================================================
import os, sys, urllib.request

try:
    from google.colab import files  # type: ignore
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    print("🚀 Detected Google Colab. Setting up environment...")
    _repo = "/content/Indie-Comic"
    if not os.path.exists(_repo):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
            "https://github.com/Cyberpunk-San/Indie-Comic.git", _repo], check=True)
    
    # Run the setup script in the main kernel context
    setup_file = f"{_repo}/indie_comic_pipeline/colab_setup.py"
    exec(open(setup_file).read(), globals())
else:
    print("💻 Detected Local Jupyter. Setting up path...")
    _candidates = [
        os.path.join(os.getcwd(), "colab_setup.py"),
        os.path.join(os.getcwd(), "indie_comic_pipeline", "colab_setup.py"),
    ]
    _found = next((p for p in _candidates if os.path.exists(p)), None)
    if _found:
        exec(open(_found).read(), globals())
    else:
        print("⚠️ colab_setup.py not found — run from repo root")

## 📦 1. Export Formats & Run RLHF Optimization Loop

In [ ]:
import os
from PIL import Image
from comic_exporter import ComicExporter
from core.feedback import RLHFFeedbackLoop
from core.optimizer import SystemOptimizer

exporter = ComicExporter(output_dir="outputs/comics")
mock_page = {'page_num': 1, 'page_image': Image.new('RGB', (800, 1200), 'white'), 'panels': []}
pages = [mock_page]

cbz = exporter.export_cbz(pages, title="FinalComic")
print("Exported CBZ:", cbz)

# Initialize telemetry
feedback_path = "outputs/comics/test_rlhf_feedback.json"
feedback = RLHFFeedbackLoop(feedback_path=feedback_path)
feedback.add_panel_feedback(panel_id=1, rating=5, comment="Excellent style consistency!", prompt_used="...", generation_backend="sdxl")

# Run optimizer
optimizer = SystemOptimizer(feedback_loop=feedback, settings_path="config/settings.yaml")
adjusts = optimizer.optimize_system_parameters()

print("System Optimization Recommendations:")
print(adjusts)